In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from joblib import dump

In [3]:
column_names = (
    ["unit", "cycle"]
    + ["setting_1", "setting_2", "setting_3"]
    + [f"sensor_{i}" for i in range(1, 22)]
)

train_df = pd.read_csv(
    "../data/raw/train_FD001.txt",
    sep=r"\s+",
    header=None,
    names=column_names
)
train_df.head()
max_cycles = train_df.groupby("unit")["cycle"].transform("max")

train_df["RUL"] = max_cycles - train_df["cycle"]

sensor_columns = [f"sensor_{i}" for i in range(1, 22)]
sensor_std = train_df[sensor_columns].std()
sensor_std.sort_values()
low_variance_sensors = sensor_std[sensor_std < 0.01]
selected_sensors = [
    sensor for sensor in sensor_columns
    if sensor not in low_variance_sensors.index
]
train_df = train_df[
    ["unit", "cycle", "setting_1", "setting_2", "setting_3"]
    + selected_sensors
    + ["RUL"]
]

print(train_df.shape)
print(train_df.columns.tolist())

(20631, 20)
['unit', 'cycle', 'setting_1', 'setting_2', 'setting_3', 'sensor_2', 'sensor_3', 'sensor_4', 'sensor_7', 'sensor_8', 'sensor_9', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_17', 'sensor_20', 'sensor_21', 'RUL']


In [4]:
train_df = train_df.sort_values(["unit", "cycle"]).reset_index(drop=True)

In [5]:
#rolling mean: it tells the model the recent average was ---- giving it temporal context
windows = [5, 10, 20]
rolling_mean_features = {}

for window in windows:
    for sensor in selected_sensors:

        rolling_mean_features[f"{sensor}_roll_mean_{window}"] = (
            train_df
            .groupby("unit")[sensor]
            .transform(
                lambda x: x.rolling(
                    window=window,
                    min_periods=1
                ).mean()
            )
        )

rolling_mean_df = pd.DataFrame(
    rolling_mean_features,
    index=train_df.index
)

train_df = pd.concat(
    [train_df, rolling_mean_df],
    axis=1
)

In [6]:
"""
rolling std: how unstable the sensor has recently become
A degrading engine may not only show a changing mean sensor value.
Some signals may become increasingly unstable.
"""

rolling_std_features = {}

for window in windows:
    for sensor in selected_sensors:

        rolling_std_features[f"{sensor}_roll_std_{window}"] = (
            train_df
            .groupby("unit")[sensor]
            .transform(
                lambda x: x.rolling(
                    window=window,
                    min_periods=2
                ).std()
            )
        )

# Add all rolling std features at once
rolling_std_df = pd.DataFrame(
    rolling_std_features,
    index=train_df.index
)

train_df = pd.concat(
    [train_df, rolling_std_df],
    axis=1
)
roll_std_cols = [
    col for col in train_df.columns
    if "_roll_std_" in col
]

"""
At the first cycle of each engine there isn't enough history 
to calculate standard deviation.
"""

train_df[roll_std_cols] = train_df[roll_std_cols].fillna(0)

In [7]:
"""
sensor difference feature: current sensor value - previous sensor value
This tells the model the direction and magnitude of recent change.
"""

diff_features = {}

for sensor in selected_sensors:

    diff_features[f"{sensor}_diff"] = (
        train_df
        .groupby("unit")[sensor]
        .diff()
    )

# Add all difference features at once
diff_df = pd.DataFrame(
    diff_features,
    index=train_df.index
)

train_df = pd.concat(
    [train_df, diff_df],
    axis=1
)

diff_cols = [
    col for col in train_df.columns
    if col.endswith("_diff")
]

train_df[diff_cols] = train_df[diff_cols].fillna(0)

In [8]:
train_df.shape

(20631, 118)

In [9]:
train_df.head()

,unit,cycle,setting_1,setting_2,setting_3,sensor_2,sensor_3,sensor_4,sensor_7,sensor_8,...,sensor_8_diff,sensor_9_diff,sensor_11_diff,sensor_12_diff,sensor_13_diff,sensor_14_diff,sensor_15_diff,sensor_17_diff,sensor_20_diff,sensor_21_diff
0,1,1,-0.0007,-0.0004,100.0,641.82,1589.70,1400.60,554.36,2388.06,...,0.00,0.00,0.00,0.00,0.00,0.00,0.0000,0.0,0.00,0.0000
1,1,2,0.0019,-0.0003,100.0,642.15,1591.82,1403.14,553.75,2388.04,...,-0.02,-2.12,0.02,0.62,0.05,-7.13,0.0123,0.0,-0.06,0.0046
2,1,3,-0.0043,0.0003,100.0,642.35,1587.99,1404.20,554.26,2388.08,...,0.04,8.87,-0.22,0.14,-0.04,1.74,-0.0140,-2.0,-0.05,-0.0794
3,1,4,0.0007,0.0000,100.0,642.35,1582.79,1401.87,554.45,2388.11,...,0.03,-3.46,-0.14,0.44,0.05,0.60,-0.0496,2.0,-0.07,0.0297
4,1,5,-0.0019,-0.0002,100.0,642.37,1582.85,1406.22,554.00,2388.06,...,-0.05,5.67,0.15,-0.67,-0.04,-0.03,0.0612,1.0,0.02,0.0305


In [10]:
engineered_cols = [
    col for col in train_df.columns
    if "roll" in col or "diff" in col
]

len(engineered_cols)

98

In [11]:
train_df.isna().sum().sum()

np.int64(0)

In [12]:
train_df.to_csv(
    "../data/processed/train_features.csv",
    index=False
)

In [13]:
exclude_cols = ["unit", "cycle", "RUL", "setting_1", "setting_2", "setting_3"]

feature_cols = [
    col for col in train_df.columns
    if col not in exclude_cols
]

X = train_df[feature_cols]
y = train_df["RUL"]

X.shape

(20631, 112)

In [14]:

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        train_df,
        groups=train_df["unit"]
    )
)

train_data = train_df.iloc[train_idx]
val_data = train_df.iloc[val_idx]


set(train_data["unit"]).intersection(
    set(val_data["unit"])
)

set()

In [15]:
X_train = train_data[feature_cols]
y_train = train_data["RUL"]

X_val = val_data[feature_cols]
y_val = val_data["RUL"]

X_train.shape, X_val.shape

((16561, 112), (4070, 112))

In [16]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)

dump(
    scaler,
    "../models/feature_engineered_scaler.joblib"
)

['../models/feature_engineered_scaler.joblib']